# newsrec_bench — Kaggle runner (V10)

Code nằm trong `training/` của repo; notebook này chỉ clone repo, import các script và chạy.

1. Settings: bật GPU + Internet. Add Input: dataset có `MINDsmall_train` và `MINDsmall_dev`.
2. Sửa `REPO_URL` / `BRANCH` bên dưới. Repo private → thêm Kaggle Secret `GITHUB_TOKEN`.
3. Run All. Artifact nằm ở `/kaggle/working/{news_emb.npz,v10_final}` — tải về và đặt vào `inference/artifacts/`.
4. Cuối notebook tự upload artifact lên Hugging Face Hub (cần Kaggle Secret `HF_TOKEN` quyền write).

In [1]:
REPO_URL = 'https://github.com/nguyenpnguyen/recsys-project.git'
BRANCH = 'main'
MODELS = ['nrms', 'naml', 'fastformer', 'caum', 'lightgcn', 'llmenc_ca',
          'supermodel', 'super_no_diff', 'super_no_ca', 'super_no_graph', 'super_no_cl', 'super_no_bge',
          'popularity', 'bge_zeroshot', 'bge_zs_pop']
TRAIN_ARGS = ['--max-epochs', '12', '--min-epochs', '3', '--patience', '2', '--min-delta', '0.001',
              '--dim', '64', '--seed', '0', '--batch-size', '64', '--eval-batch', '32',
              '--lr', '1e-3', '--weight-decay', '1e-5', '--dev-ratio', '0.10']
BGE_MODEL = 'BAAI/bge-small-en-v1.5'
QWEN_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
QWEN_MODELS = ['llmenc_ca', 'supermodel', 'bge_zeroshot', 'bge_zs_pop']  # các model dùng embedding

In [2]:
import os, sys, glob, subprocess
from pathlib import Path

ON_KAGGLE = os.path.isdir('/kaggle/working')
if ON_KAGGLE:
    REPO = Path('/kaggle/working/repo'); WORK = Path('/kaggle/working')
    if not REPO.exists():
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            url = url.replace('https://', f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
        except Exception:
            pass  # public repo
        subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, str(REPO)], check=True)
    hits = glob.glob('/kaggle/input/**/MINDsmall_train/news.tsv', recursive=True)
    assert hits, 'Add Input: dataset chứa MINDsmall_train/ và MINDsmall_dev/'
    DATA = Path(hits[0]).parent.parent
else:  # chạy local từ training/
    REPO = Path.cwd().parent; WORK = REPO / 'inference' / 'artifacts'; DATA = REPO / 'dataset'

sys.path.insert(0, str(REPO / 'training'))
MIND_TRAIN, MIND_DEV = DATA / 'MINDsmall_train', DATA / 'MINDsmall_dev'
NEWS_EMB, OUT = WORK / 'news_emb.npz', WORK / 'v10_final'
QWEN_EMB, QWEN_OUT = WORK / 'news_emb_qwen3.npz', WORK / 'v10_qwen3'
assert (MIND_DEV / 'behaviors.tsv').exists(), MIND_DEV

import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('DATA =', DATA, '| OUT =', OUT)

Cloning into '/kaggle/working/repo'...


torch 2.10.0+cu128 | GPU: Tesla T4
DATA = /kaggle/input/datasets/thinhhuynh3108/mindsmall | OUT = /kaggle/working/v10_final


### 1. BGE embedding (title + abstract, union train + dev)

In [3]:
import numpy as np, llm_embed
if not (NEWS_EMB.exists() and str(np.load(NEWS_EMB)['model_name']) == BGE_MODEL):
    llm_embed.main(['--news', str(MIND_TRAIN / 'news.tsv'), str(MIND_DEV / 'news.tsv'),
                    '--out', str(NEWS_EMB), '--model', BGE_MODEL])
print('NEWS_EMB =', NEWS_EMB)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

Saved 65238 embeddings of dim 384 -> /kaggle/working/news_emb.npz
NEWS_EMB = /kaggle/working/news_emb.npz


### 2. Train + test 12 model

In [4]:
import bench
bench.main(['--mind-train', str(MIND_TRAIN), '--mind-dev', str(MIND_DEV), '--news-emb', str(NEWS_EMB),
            '--out', str(OUT), '--models', *MODELS, *TRAIN_ARGS])

  embedding coverage: 100.000% (PAD excluded)
  chronological split: 126,695 train_core / 30,270 validation / 73,152 official test impressions

===== nrms (baseline) =====
  epoch 01 step=2,962 loss=1.4622 val_auc=0.6036 val_ndcg@10=0.3351
  epoch 02 step=5,924 loss=1.3832 val_auc=0.6164 val_ndcg@10=0.3469
  epoch 03 step=8,886 loss=1.3482 val_auc=0.6256 val_ndcg@10=0.3518
  epoch 04 step=11,848 loss=1.3269 val_auc=0.6314 val_ndcg@10=0.3581
  epoch 05 step=14,810 loss=1.3127 val_auc=0.6365 val_ndcg@10=0.3614
  epoch 06 step=17,772 loss=1.3025 val_auc=0.6327 val_ndcg@10=0.3592
  epoch 07 step=20,734 loss=1.2921 val_auc=0.6305 val_ndcg@10=0.3576

===== naml (baseline) =====
  epoch 01 step=2,962 loss=1.4197 val_auc=0.6138 val_ndcg@10=0.3488
  epoch 02 step=5,924 loss=1.3715 val_auc=0.6235 val_ndcg@10=0.3528
  epoch 03 step=8,886 loss=1.3489 val_auc=0.6176 val_ndcg@10=0.3515
  epoch 04 step=11,848 loss=1.3319 val_auc=0.6228 val_ndcg@10=0.3594
  epoch 05 step=14,810 loss=1.3197 val_auc=0.6

### 3. Kiểm tra artifact

In [5]:
bench.verify(OUT)

✅ verified nrms
✅ verified naml
✅ verified fastformer
✅ verified caum
✅ verified lightgcn
✅ verified llmenc_ca
✅ verified supermodel
✅ verified super_no_diff
✅ verified super_no_ca
✅ verified super_no_graph
✅ verified super_no_cl
✅ verified super_no_bge
✅ verified popularity
✅ verified bge_zeroshot
✅ verified bge_zs_pop
✅ V10: 15 rows, chronological split, epoch curves, and checkpoint/test artifacts verified
# V10 final benchmark

| model | group | control | best_epoch | epochs_trained | train_steps | test_auc | test_mrr | test_ndcg@5 | test_ndcg@10 | delta_auc | delta_mrr | delta_ndcg@5 | delta_ndcg@10 | params | train_s | infer_impr/s |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| bge_zs_pop | no-training | — | 1 | 1 | 0 | 0.6769 | 0.3349 | 0.3727 | 0.4330 | — | — | — | — | 0 | 8.379975193000064 | 3598 |
| popularity | no-training | — | 1 | 1 | 0 | 0.6533 | 0.3114 | 0.3402 | 0.4026 | — | — | — | — | 0 | 7.6619

### 3b. Benchmark lại các model dùng embedding với Qwen3-Embedding-0.6B

In [ ]:
if not (QWEN_EMB.exists() and str(np.load(QWEN_EMB)['model_name']) == QWEN_MODEL):
    llm_embed.main(['--news', str(MIND_TRAIN / 'news.tsv'), str(MIND_DEV / 'news.tsv'),
                    '--out', str(QWEN_EMB), '--model', QWEN_MODEL, '--batch-size', '32'])
bench.main(['--mind-train', str(MIND_TRAIN), '--mind-dev', str(MIND_DEV), '--news-emb', str(QWEN_EMB),
            '--out', str(QWEN_OUT), '--models', *QWEN_MODELS, *TRAIN_ARGS])
bench.verify(QWEN_OUT)

### 4. Upload artifact lên Hugging Face Hub

In [ ]:
HF_REPO = 'nguyenpn/recsys-artifacts'
from huggingface_hub import HfApi
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = None  # local: dùng token từ `hf auth login`
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(repo_id=HF_REPO, folder_path=str(WORK), allow_patterns=['news_emb.npz', 'v10_final/**', 'news_emb_qwen3.npz', 'v10_qwen3/**'])
print('Uploaded ->', f'https://huggingface.co/{HF_REPO}')